In [6]:
import sys
import re

def main():
    # Read all lines from standard input
    input_data = sys.stdin.read().splitlines()
    if not input_data:
        return

    try:
        N = int(input_data[0].strip())
    except ValueError:
        return

    # The regular expression pattern for a token
    pattern = re.compile(r"[A-Za-z]+(?:['-][A-Za-z]+)*")

    for i in range(1, N + 1):
        if i < len(input_data):
            line = input_data[i]
        else:
            line = ""
        tokens = pattern.findall(line)
        if tokens:
            print(f"{len(tokens)} {' '.join(tokens)}")
        else:
            print("0")

if __name__ == '__main__':
    main()

In [8]:
import io

# Sample input from the problem description
sample_input = """4
The dog's tail wagged -- twice!
state-of-the-art NLP systems, circa 2026.
12345 ... ???
Don't re-use rock--roll O'Neill's x"""

# Backup sys.stdin and redirect it to our sample input
original_stdin = sys.stdin
sys.stdin = io.StringIO(sample_input)

try:
    print("--- Running Test with Sample Input ---")
    main()
finally:
    # Restore original stdin
    sys.stdin = original_stdin

--- Running Test with Sample Input ---
5 The dog's tail wagged twice
4 state-of-the-art NLP systems circa
0
6 Don't re-use rock roll O'Neill's x


In [10]:
import sys

def main_fsa():
    input_data = sys.stdin.read().splitlines()
    if not input_data:
        return

    # Parse first line: n (num states), k (alphabet size)
    parts_1 = input_data[0].split()
    n = int(parts_1[0])
    k = int(parts_1[1])

    # Parse second line: alphabet
    alphabet = set(input_data[1].split())

    # Parse third line: start state
    start_state = int(input_data[2].strip())

    # Parse fourth line: accepting states
    accept_parts = input_data[3].split()
    num_accepting = int(accept_parts[0])
    accepting_states = set(int(x) for x in accept_parts[1:])

    # Parse fifth line: number of transitions t
    t = int(input_data[4].strip())

    # Build transitions structure: (state, char) -> set of next states
    transitions = {}
    for i in range(5, 5 + t):
        p_str, c, r_str = input_data[i].split()
        p = int(p_str)
        r = int(r_str)
        if (p, c) not in transitions:
            transitions[(p, c)] = set()
        transitions[(p, c)].add(r)

    # Parse queries count Q
    q_idx = 5 + t
    Q = int(input_data[q_idx].strip())

    # Process Q queries
    for i in range(1, Q + 1):
        query = input_data[q_idx + i]
        if query == "#":
            query = ""  # Empty string ε

        # Track current set of states
        current_states = {start_state}

        # Advance set of states symbol by symbol
        possible = True
        for char in query:
            next_states = set()
            for state in current_states:
                if (state, char) in transitions:
                    next_states.update(transitions[(state, char)])
            current_states = next_states
            if not current_states:
                possible = False
                break

        if possible and (current_states & accepting_states):
            print("ACCEPT")
        else:
            print("REJECT")

if __name__ == '__main__':
    # Test block to verify with sample input
    import io
    sample_fsa_input = """5 3
b a !
0
1 4
5
0 b 1
1 a 2
2 a 3
3 a 3
3 ! 4
5
baa!
baaa!
ba!
b
baa"""

    print("--- Running Test with Sample FSA Input ---")
    original_stdin = sys.stdin
    sys.stdin = io.StringIO(sample_fsa_input)
    try:
        main_fsa()
    finally:
        sys.stdin = original_stdin

--- Running Test with Sample FSA Input ---
ACCEPT
ACCEPT
REJECT
REJECT
REJECT


In [11]:
import sys

def is_vowel(char):
    return char in 'aeiou'

def is_consonant(char):
    return char.isalpha() and char not in 'aeiou'

def count_vowel_groups(stem):
    # A vowel group is a maximal run of one or more consecutive vowels
    groups = 0
    in_vowel = False
    for char in stem:
        if is_vowel(char):
            if not in_vowel:
                groups += 1
                in_vowel = True
        else:
            in_vowel = False
    return groups

def is_doublable(stem):
    # A stem is doublable when all four of the following hold:
    # 1) has at least 3 letters;
    if len(stem) < 3:
        return False
    # 2) its last three letters are consonant, vowel, consonant in that order;
    if not (is_consonant(stem[-3]) and is_vowel(stem[-2]) and is_consonant(stem[-1])):
        return False
    # 3) its final letter is not w, x or y;
    if stem[-1] in 'wxy':
        return False
    # 4) it contains exactly one vowel group.
    if count_vowel_groups(stem) != 1:
        return False
    return True

def inflect(stem, feature):
    # Rule set A — features N+PL and V+3SG
    if feature in ('N+PL', 'V+3SG'):
        # A1: ends in s, x, z, ch or sh -> add es
        if stem.endswith(('s', 'x', 'z', 'ch', 'sh')):
            return stem + 'es'
        # A2: ends in a consonant followed by o -> add es
        if len(stem) >= 2 and is_consonant(stem[-2]) and stem.endswith('o'):
            return stem + 'es'
        # A3: ends in a consonant followed by y -> replace y with ies
        if len(stem) >= 2 and is_consonant(stem[-2]) and stem.endswith('y'):
            return stem[:-1] + 'ies'
        # A4: otherwise -> add s
        return stem + 's'

    # Rule set B — feature V+PROG (suffix -ing)
    elif feature == 'V+PROG':
        # B1: has at least 3 letters and ends in a consonant followed by e -> drop e, add ing
        if len(stem) >= 3 and is_consonant(stem[-2]) and stem.endswith('e'):
            return stem[:-1] + 'ing'
        # B2: is doublable -> double last letter, add ing
        if is_doublable(stem):
            return stem + stem[-1] + 'ing'
        # B3: otherwise -> add ing
        return stem + 'ing'

    # Rule set C — features V+PAST (-ed), ADJ+COMP (-er) and ADJ+SUP (-est)
    elif feature in ('V+PAST', 'ADJ+COMP', 'ADJ+SUP'):
        suffix_map = {'V+PAST': 'ed', 'ADJ+COMP': 'er', 'ADJ+SUP': 'est'}
        suffix = suffix_map[feature]

        # C1: ends in e -> add d / r / st
        if stem.endswith('e'):
            short_suffix = {'ed': 'd', 'er': 'r', 'est': 'st'}[suffix]
            return stem + short_suffix
        # C2: ends in a consonant followed by y -> replace y with i, then add ed / er / est
        if len(stem) >= 2 and is_consonant(stem[-2]) and stem.endswith('y'):
            return stem[:-1] + 'i' + suffix
        # C3: is doublable -> double the last letter, then add ed / er / est
        if is_doublable(stem):
            return stem + stem[-1] + suffix
        # C4: otherwise -> add ed / er / est
        return stem + suffix

def main_morphology():
    input_data = sys.stdin.read().splitlines()
    if not input_data:
        return

    # Parse irregular entries
    L = int(input_data[0].strip())
    lexicon = {}
    for i in range(1, L + 1):
        stem, feature, form = input_data[i].split()
        lexicon[(stem, feature)] = form

    # Parse queries
    q_idx = L + 1
    Q = int(input_data[q_idx].strip())

    for i in range(1, Q + 1):
        stem, feature = input_data[q_idx + i].split()
        if (stem, feature) in lexicon:
            print(lexicon[(stem, feature)])
        else:
            print(inflect(stem, feature))

if __name__ == '__main__':
    # Test block with sample input
    import io
    sample_morph_input = """3
goose N+PL geese
go V+PAST went
good ADJ+COMP better
8
cat N+PL
church N+PL
baby N+PL
stop V+PROG
make V+PROG
carry V+PAST
goose N+PL
go V+PAST"""

    print("--- Running Test with Sample Morphology Input ---")
    original_stdin = sys.stdin
    sys.stdin = io.StringIO(sample_morph_input)
    try:
        main_morphology()
    finally:
        sys.stdin = original_stdin

--- Running Test with Sample Morphology Input ---
cats
churches
babies
stopping
making
carried
geese
went


In [14]:
def parse_morphology():
    # The morphological rules from Problem 3
    # We reuse is_vowel, is_consonant, count_vowel_groups, is_doublable, and inflect

    import sys
    input_data = sys.stdin.read().splitlines()
    if not input_data:
        return

    # Read Lexicon
    S = int(input_data[0].strip())
    lexicon = [] # list of (stem, pos)
    for i in range(1, S + 1):
        parts = input_data[i].split()
        lexicon.append((parts[0], parts[1]))

    # Read Queries
    q_idx = S + 1
    Q = int(input_data[q_idx].strip())

    # Map parts of speech to valid grammatical features as expected by inflect
    pos_features = {
        'N': ['PL'],
        'V': ['3SG', 'PROG', 'PAST'],
        'ADJ': ['COMP', 'SUP']
    }

    for i in range(1, Q + 1):
        surface = input_data[q_idx + i].strip()
        valid_analyses = []

        # Check each lexicon entry
        for stem, pos in lexicon:
            # Rule: A surface form that is itself a stem is a valid analysis with no features
            if surface == stem:
                valid_analyses.append(f"{stem}+{pos}")

            # Check all valid morphological inflections for this POS
            features = pos_features.get(pos, [])
            for feature in features:
                full_feature_key = f"{pos}+{feature}"
                if inflect(stem, full_feature_key) == surface:
                    valid_analyses.append(f"{stem}+{pos}+{feature}")

        if valid_analyses:
            # De-duplicate, sort lexicographically, and join
            unique_analyses = sorted(list(set(valid_analyses)))
            print(", ".join(unique_analyses))
        else:
            print("UNKNOWN")

# Quick validation with the sample input
import io
sample_parser_input = """5
cat N
watch N
watch V
study V
big ADJ
8
cats
watches
watching
studied
bigger
cat
biggest
elephants"""

print("--- Running Test with Sample Parser Input ---")
original_stdin = sys.stdin
sys.stdin = io.StringIO(sample_parser_input)
try:
    parse_morphology()
finally:
    sys.stdin = original_stdin

--- Running Test with Sample Parser Input ---
cat+N+PL
watch+N+PL, watch+V+3SG
watch+V+PROG
study+V+PAST
big+ADJ+COMP
cat+N
big+ADJ+SUP
UNKNOWN


In [15]:
import sys

def is_consonant(word, i):
    # A consonant is a letter other than a, e, i, o, u,
    # and other than y preceded by a consonant.
    # A y at the very start of a word is a consonant.
    val = word[i]
    if val in 'aeiou':
        return False
    if val == 'y':
        if i == 0:
            return True
        else:
            return not is_consonant(word, i - 1)
    return True

def get_cv_sequence(word):
    # Returns a list of 'C' or 'V' characters representing the consonant/vowel structure
    seq = []
    for i in range(len(word)):
        if is_consonant(word, i):
            if not seq or seq[-1] != 'C':
                seq.append('C')
        else:
            if not seq or seq[-1] != 'V':
                seq.append('V')
    return seq

def get_measure(word):
    # Every word can be written in the form [C] (VC)^m [V]
    # m is the number of VC pairs in the sequence
    seq = get_cv_sequence(word)
    # Count VC combinations
    m = 0
    i = 0
    if i < len(seq) and seq[i] == 'C':
        i += 1
    while i + 1 < len(seq):
        if seq[i] == 'V' and seq[i+1] == 'C':
            m += 1
            i += 2
        else:
            i += 1
    return m

def has_vowel(word):
    # Condition *v*: the stem contains a vowel
    for i in range(len(word)):
        if not is_consonant(word, i):
            return True
    return False

def ends_double_consonant(word):
    # Condition *d: the stem ends with a double consonant
    if len(word) < 2:
        return False
    return word[-1] == word[-2] and is_consonant(word, len(word) - 1)

def cvc_end(word):
    # Condition *o: ends consonant-vowel-consonant,
    # and the final consonant is not w, x, or y
    if len(word) < 3:
        return False
    return (is_consonant(word, len(word) - 3) and
            not is_consonant(word, len(word) - 2) and
            is_consonant(word, len(word) - 1) and
            word[-1] not in 'wxy')

def step_1a(word):
    if word.endswith('sses'):
        return word[:-4] + 'ss'
    elif word.endswith('ies'):
        return word[:-3] + 'i'
    elif word.endswith('ss'):
        return word
    elif word.endswith('s'):
        return word[:-1]
    return word

def step_1b_cleanup(stem):
    if stem.endswith('at') or stem.endswith('bl') or stem.endswith('iz'):
        return stem + 'e'
    elif ends_double_consonant(stem) and stem[-1] not in 'lsz':
        return stem[:-1]
    elif get_measure(stem) == 1 and cvc_end(stem):
        return stem + 'e'
    return stem

def step_1b(word):
    if word.endswith('eed'):
        stem = word[:-3]
        if get_measure(stem) > 0:
            return stem + 'ee'
        return word
    elif word.endswith('ed'):
        stem = word[:-2]
        if has_vowel(stem):
            return step_1b_cleanup(stem)
        return word
    elif word.endswith('ing'):
        stem = word[:-3]
        if has_vowel(stem):
            return step_1b_cleanup(stem)
        return word
    return word

def step_1c(word):
    if word.endswith('y') and len(word) > 1:
        stem = word[:-1]
        if has_vowel(stem):
            return stem + 'i'
    return word

def stem_word(word):
    w = step_1a(word)
    w = step_1b(w)
    w = step_1c(w)
    return w

def main_porter():
    import sys
    input_data = sys.stdin.read().splitlines()
    if not input_data:
        return
    try:
        Q = int(input_data[0].strip())
    except ValueError:
        return
    for i in range(1, Q + 1):
        if i < len(input_data):
            word = input_data[i].strip().lower()
            print(stem_word(word))

# Verification with sample input
import io
sample_porter_input = """8
caresses
ponies
cats
feed
agreed
motoring
hopping
happy"""

print("--- Running Test with Sample Porter Stemmer Input ---")
original_stdin = sys.stdin
sys.stdin = io.StringIO(sample_porter_input)
try:
    main_porter()
finally:
    sys.stdin = original_stdin

--- Running Test with Sample Porter Stemmer Input ---
caress
poni
cat
feed
agree
motor
hop
happi
